# ESM2-35M generation malleability (Reviewer R1.4, Phase 1)

Apples-to-apples with the ProteinMPNN designs: fine-tune AlkSecESM35M, then compute the **per-position amino-acid probability map** (base vs fine-tuned) for the 5 templates with the largest ProteinMPNN surface shift. Phase 1 answers *does the fine-tune change what ESM would place at each position?*; Phase 2 (optional cell at the end) actually generates sequences by iterative in-filling.

Upload **`esm35m_generation_inputs.zip`**; set Runtime -> **T4 GPU**.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.45" "datasets>=2.20" "accelerate>=0.33" safetensors 2>/dev/null
import transformers, torch
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 2. Upload & unzip

In [ ]:
import os, io, zipfile
from google.colab import files
up = files.upload()
name = [n for n in up if n.endswith('.zip')][0]
with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall('/content/esm_gen')
os.chdir('/content/esm_gen')
import sys; sys.path.insert(0, 'scripts')
os.makedirs('gen_out', exist_ok=True)
print(sorted(os.listdir()))

## 3. Config

In [ ]:
BASE_MODEL = "facebook/esm2_t12_35M_UR50D"
EPOCHS = 30   # dose curve saturates by ~30; bump to preempt under-training doubts
LR = 5e-5

## 4. Fine-tune AlkSecESM35M + the neutralophile control
Same recipe/data as the scoring arm. The matched neutralophile-control fine-tune is the specificity check: a cohort-specific surface shift should appear in AlkSecESM35M but not (or far less) in NeuSecESM35M.

In [ ]:
ARMS = {'AlkSecESM35M': 'alkaline_case', 'NeuSecESM35M': 'alkaline_neu'}
for name, stem in ARMS.items():
    print(f'\n===== training {name} =====')
    !python scripts/train_esm2_mlm.py \
      --train_csv data/{stem}_train.csv --val_csv data/{stem}_val.csv \
      --out_dir runs/{name} --epochs {EPOCHS} --learning_rate {LR}

## 5. Phase 1 — per-position probability maps (base vs fine-tuned)
For each template, mask each position and read the model's distribution over the 20 amino acids, for both models. Saved as one `.npz` per template.

In [ ]:
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from esm_generation import esm_position_distributions, CANONICAL_AA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tmpl = pd.read_csv('gen_templates.csv')
MODELS = {'base': BASE_MODEL, 'AlkSecESM35M': 'runs/AlkSecESM35M',
          'NeuSecESM35M': 'runs/NeuSecESM35M'}
loaded = {}
for tag, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForMaskedLM.from_pretrained(path).to(device).eval()
    loaded[tag] = (tok, mdl)

for _, r in tmpl.iterrows():
    seq = r['wt_sequence']
    mats = {}
    for tag, (tok, mdl) in loaded.items():
        mats[tag] = esm_position_distributions(seq, tok, mdl, device)
        print(f"{r['uniprot_id']} {tag}: {mats[tag].shape}")
    np.savez(f"gen_out/{r['uniprot_id']}_probs.npz",
             seq=seq, aa_order=np.array(list(CANONICAL_AA)),
             base=mats['base'], AlkSecESM35M=mats['AlkSecESM35M'],
             NeuSecESM35M=mats['NeuSecESM35M'])

## 6. Download Phase-1 maps
Unzip into `outputs/esm35m_continual_pretraining/generation/` and run `python paper_code/08_pca_figures/esm_design_heatmaps.py` locally.

In [ ]:
import shutil
shutil.make_archive('/content/esm35m_generation_phase1', 'zip', 'gen_out')
from google.colab import files
files.download('/content/esm35m_generation_phase1.zip')

---
## 7. (Optional) Phase 2 — generate sequences by iterative in-filling
Run only after inspecting the Phase-1 heatmaps. Gibbs-style masked in-filling, T=0.1, 8 designs/template, base vs fine-tuned. Saves designed sequences.

In [ ]:
from esm_generation import make_esm_predict_fn, iterative_infill
N_DESIGNS, N_PASSES, TEMP = 8, 1, 0.1
rows = []
for _, r in tmpl.iterrows():
    seq = r['wt_sequence']
    for tag, (tok, mdl) in loaded.items():
        pf = make_esm_predict_fn(tok, mdl, device)
        for k in range(N_DESIGNS):
            des = iterative_infill(seq, pf, n_passes=N_PASSES, temperature=TEMP, seed=k)
            rows.append(dict(uniprot_id=r['uniprot_id'], model=tag, sample_idx=k, sequence=des))
    print(r['uniprot_id'], 'done')
import pandas as pd
pd.DataFrame(rows).to_csv('gen_out/esm_designs.csv', index=False)
shutil.make_archive('/content/esm35m_generation_designs', 'zip', 'gen_out')
files.download('/content/esm35m_generation_designs.zip')